# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

In [2]:
import pandas as pd
import numpy as np
import os

# --- Rebuild baseline reason codes (from ML-07) ---
visibility_bar = df["impressions_90d"].quantile(0.5)

def reason_code(row):
    if row["trend_direction"] == "down" and row["impressions_90d"] >= visibility_bar:
        return "stale_declining_visible" if row["days_since_last_update"] >= 180 else "declining_visible_fresh"
    if row["impressions_90d"] >= visibility_bar and row["days_since_last_update"] >= 180:
        return "stale_visible_stable"
    return "low_priority"

df["reason_code"] = df.apply(reason_code, axis=1)

# --- Archetype → action mapping (from clustering interpretation, ML-08/09) ---
# champions: high sessions, high engagement -> protect + refresh (paper Finding #2/#4: best pages decay too if left stale)
# hidden gems: low traffic, above-median engagement -> improve (expand, don't prune)
# weak/no-demand: near-zero impressions -> monitor, do not invest rewrite hours
# stale_declining_visible: still visible, losing ground, old -> rewrite
# declining_visible_fresh: losing ground despite recent update -> improve (investigate cause, not just rewrite)
# stale_visible_stable: old, visible, not yet declining -> monitor (early warning)
# low_priority: default bucket -> monitor

low_traffic = df["sessions_90d"] <= df["sessions_90d"].quantile(0.25)
hidden_gem = low_traffic & (df["engagement_rate"] >= df["engagement_rate"].median())
champion = (df["sessions_90d"] >= df["sessions_90d"].quantile(0.90)) & \
           (df["engagement_rate"] >= df["engagement_rate"].median())
weak_demand = df["impressions_90d"] <= df["impressions_90d"].quantile(0.10)

def archetype(row):
    if champion[row.name]: return "champion"
    if hidden_gem[row.name]: return "hidden_gem"
    if weak_demand[row.name]: return "weak_no_demand"
    return "standard"

df["archetype"] = df.apply(archetype, axis=1)

action_map = {
    "stale_declining_visible": "rewrite",
    "declining_visible_fresh": "improve",
    "stale_visible_stable": "monitor",
    "low_priority": "monitor",
}
df["action"] = df["reason_code"].map(action_map)

# archetype can override/refine the reason-code action
df.loc[df["archetype"] == "champion", "action"] = "protect_and_refresh"
df.loc[(df["archetype"] == "hidden_gem") & (df["action"] == "monitor"), "action"] = "improve"
df.loc[df["archetype"] == "weak_no_demand", "action"] = "monitor"

# --- Priority score for ranking within action groups (readable, no fitted weights) ---
stale = (df["days_since_last_update"] >= 180).astype(int)
declining = (df["trend_direction"] == "down").astype(int)
df["priority_score"] = declining * (1 + stale) * df["impressions_90d"]

priority_order = ["rewrite", "protect_and_refresh", "improve", "monitor"]
df["action"] = pd.Categorical(df["action"], categories=priority_order, ordered=True)
ranked_queue = df.sort_values(["action", "priority_score"], ascending=[True, False]).reset_index(drop=True)

print(ranked_queue["action"].value_counts())
print(ranked_queue[["content_id", "action", "reason_code", "archetype", "priority_score"]].head(15))

action
improve                14907
monitor                12065
protect_and_refresh     3015
rewrite                   13
Name: count, dtype: int64
              content_id               action              reason_code  \
0   content_7368877ea310              rewrite  stale_declining_visible   
1   content_1bfaa38ff26c              rewrite  stale_declining_visible   
2   content_0a91db491d14              rewrite  stale_declining_visible   
3   content_5feee3994adb              rewrite  stale_declining_visible   
4   content_c2d929d83eaa              rewrite  stale_declining_visible   
5   content_b16bd7307b39              rewrite  stale_declining_visible   
6   content_fe16a55cd13d              rewrite  stale_declining_visible   
7   content_ecb6215e79fd              rewrite  stale_declining_visible   
8   content_928af3e22c80              rewrite  stale_declining_visible   
9   content_e3ff1b093148              rewrite  stale_declining_visible   
10  content_7f116ae1f6f5             

In [8]:
# Confirm what happened to the page that dropped out of rewrite
check = df[df["content_id"] == "content_cf56e2e2e282"][
    ["content_id", "reason_code", "archetype", "action", "sessions_90d", "engagement_rate", "impressions_90d"]
]
print(check)

                 content_id              reason_code archetype  \
16751  content_cf56e2e2e282  stale_declining_visible  champion   

                    action  sessions_90d  engagement_rate  impressions_90d  
16751  protect_and_refresh           119             0.84            61678  


One page (`content_cf56e2e2e242`) that matched the baseline's `stale_declining_visible` reason code was reclassified from `rewrite` to `protect_and_refresh` because it also met the `champion` archetype threshold (top-10% sessions, above-median engagement). The champion override takes priority when both conditions apply, since protecting a still-valuable, high-traffic asset is a higher-stakes call than a routine rewrite. This page's champion status is borderline (engagement_rate 0.84%, just above the portfolio median) — flagged here for human review rather than treated as an automatic, confident classification.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Who uses this:** a content strategist or SEO editor triaging a sprint backlog — not an automated publishing system.

**What it's for:** deciding which pages get attention first, using observed 90-day structured metrics (traffic, engagement, freshness) and clustering-derived archetypes.

**Where it stops being valid:**

* This queue reflects one snapshot in time (the extract date). It goes stale as soon as new performance data lands — there's no live feedback loop.
* The clustering behind the `archetype` column was shown (in ML-09) to be substantially driven by `has_word_count`, a proxy for `content_type` — so "champion" or "hidden_gem" labels partly reflect content category, not purely behavior. Treat archetype names as a lens, not a fixed identity.
* `priority_score` uses a transparent rule, not a validated model — there is no precision@K against an independently observed future outcome, because `trend_direction` (used in scoring) is itself a defined rule, not an outcome. Using it as both a scoring input and a later "did it work" label would be circular.
* This tool has no visibility into content strategy, business priority, backlinks, or competitor moves — it can surface a candidate list, never override human judgment on any single page.
* Per the paper's own limitation: revenue tracking only covers a fraction of typical portfolios, so no dollar-value ranking is claimed here beyond the click-equivalent proxy in section 1 if you choose to add it.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [6]:
import os
print("Current working directory:", os.getcwd())
print("Contents:", os.listdir("."))
os.makedirs("work/outputs", exist_ok=True)
print("work/outputs exists now:", os.path.isdir("work/outputs"))

Current working directory: /content/flyrank-ml-internship-starter
Contents: ['DATA_USE.md', 'GUIDE.md', 'work', 'requirements.txt', 'outputs', 'submission', 'notebooks', '.gitignore', '.github', 'docs', '.git', 'AGENTS.md', 'CLAUDE.md', 'LICENSE', 'skills', 'README.md', 'SETUP.md', 'data', 'scripts']
work/outputs exists now: True


In [7]:
# Sample for human review: top 10 per action bucket, not just the global top 10
review_sample = (
    ranked_queue.groupby("action", observed=True)
    .head(10)[["content_id", "client_id", "action", "reason_code", "archetype",
               "impressions_90d", "sessions_90d", "trend_pct", "avg_position",
               "days_since_last_update", "content_type", "word_count"]]
)
review_sample.to_csv("work/outputs/human_review_sample.csv", index=False)
print(review_sample.head(10))

             content_id          client_id   action              reason_code  \
0  content_7368877ea310  client_7f2253d7e2  rewrite  stale_declining_visible   
1  content_1bfaa38ff26c  client_7f2253d7e2  rewrite  stale_declining_visible   
2  content_0a91db491d14  client_7f2253d7e2  rewrite  stale_declining_visible   
3  content_5feee3994adb  client_7f2253d7e2  rewrite  stale_declining_visible   
4  content_c2d929d83eaa  client_7f2253d7e2  rewrite  stale_declining_visible   
5  content_b16bd7307b39  client_7f2253d7e2  rewrite  stale_declining_visible   
6  content_fe16a55cd13d  client_7f2253d7e2  rewrite  stale_declining_visible   
7  content_ecb6215e79fd  client_7f2253d7e2  rewrite  stale_declining_visible   
8  content_928af3e22c80  client_7f2253d7e2  rewrite  stale_declining_visible   
9  content_e3ff1b093148  client_d029fa3a95  rewrite  stale_declining_visible   

  archetype  impressions_90d  sessions_90d  trend_pct  avg_position  \
0  standard            59472            82      

**Before acting on any recommendation, a human must check:**

* Is `avg_position` computed from a meaningful impression base? Per flyrank-data, `avg_position = 0` means no data — and per the paper's own caution, position averages from tiny impression counts (e.g. the `weak_no_demand` bucket) are statistically unreliable, not evidence of "secretly ranking well."
* Does the page belong to a client whose overall inventory is systematically older or newer than the portfolio median? (Confirmed in ML-09: client size and staleness are highly skewed — one client alone held 23% of rows.)
* Is a "decline" seasonal rather than structural? The rule can't distinguish the two.
* Was this page intentionally deprecated (redirected, retired) — if so it shouldn't be in a rewrite queue at all.

**No-go list — what should never be automated:**

* Auto-publishing or auto-editing content based on the `action` column. The queue prioritizes review order; it does not write or approve copy.
* Auto-pruning or auto-merging pages. A `monitor/weak_no_demand` label is not permission to delete — per ML-02's own cost analysis, a false prune on a hidden gem is one of the costliest possible errors.
* Treating `priority_score` as a dollar figure or committing budget/headcount automatically from it — it's a readable rule, not a validated economic model.
* Auto-labeling a page's archetype as a permanent identity fed into other systems — it's a snapshot lens, confirmed to shift with each new data pull.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [9]:
# Baseline distribution to compare future snapshots against (data drift check)
baseline_snapshot = {
    "action_counts": ranked_queue["action"].value_counts().to_dict(),
    "archetype_counts": ranked_queue["archetype"].value_counts().to_dict(),
    "median_impressions_90d": float(df["impressions_90d"].median()),
    "pct_trend_down": float((df["trend_direction"] == "down").mean()),
    "snapshot_date": pd.Timestamp.today().strftime("%Y-%m-%d"),
}
import json
with open("work/outputs/monitoring_baseline.json", "w") as f:
    json.dump(baseline_snapshot, f, indent=2)
print(baseline_snapshot)

{'action_counts': {'improve': 14907, 'monitor': 12065, 'protect_and_refresh': 3015, 'rewrite': 13}, 'archetype_counts': {'standard': 17939, 'hidden_gem': 8073, 'champion': 3015, 'weak_no_demand': 973}, 'median_impressions_90d': 731.0, 'pct_trend_down': 0.5420666666666667, 'snapshot_date': '2026-08-27'}


**Signals that the playbook has gone stale and needs a refresh, not just a re-run:**

* The share of pages trending `down` shifts meaningfully from the current 54% baseline — a big swing suggests portfolio-wide dynamics changed (algorithm update, seasonal shift) that a static rule won't adapt to.
* Action-bucket counts drift heavily from the recorded baseline (e.g. `rewrite` count doubles or evaporates) — worth checking whether that's real change or a broken upstream field.
* `has_word_count/has_search_volume` missingness rates change — since these are near-proxies for `content_type` and drive much of the archetype split (per ML-09's decision-tree finding), a shift here silently reshapes the clusters without anyone noticing.
* A new `content_type` or `main_intent` value appears that wasn't in the original 30k-row extract — the archetype rules were tuned on this specific mix and may not generalize.
* Cadence: re-run this notebook monthly at minimum, and immediately after any known GSC/GA4 tracking or reporting change, since the metrics feeding every rule here come directly from those sources.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [10]:
os.makedirs("work/outputs", exist_ok=True)

export_cols = [
    "content_id", "client_id", "content_type", "action", "reason_code", "archetype",
    "priority_score", "impressions_90d", "sessions_90d", "trend_direction", "trend_pct",
    "avg_position", "engagement_rate", "days_since_last_update", "content_age_days", "word_count"
]
ranked_queue[export_cols].to_csv("work/outputs/content_action_playbook.csv", index=False)

summary = ranked_queue.groupby("action", observed=True).agg(
    pages=("content_id", "count"),
    median_impressions=("impressions_90d", "median"),
    median_priority_score=("priority_score", "median"),
).reset_index()
summary.to_csv("work/outputs/playbook_summary_by_action.csv", index=False)

review_sample.to_csv("work/outputs/human_review_sample.csv", index=False)

import json
with open("work/outputs/monitoring_baseline.json", "w") as f:
    json.dump(baseline_snapshot, f, indent=2)

print("Exported files:")
for f in ["content_action_playbook.csv", "human_review_sample.csv", "monitoring_baseline.json", "playbook_summary_by_action.csv"]:
    path = f"work/outputs/{f}"
    print(path, "-", os.path.exists(path))

Exported files:
work/outputs/content_action_playbook.csv - True
work/outputs/human_review_sample.csv - True
work/outputs/monitoring_baseline.json - True
work/outputs/playbook_summary_by_action.csv - True


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.